<a href="https://colab.research.google.com/github/AhmedCode110/AC-MOT/blob/main/notebooks/AC_MOT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AC-MOT portable Colab runner
Select a Tesla T4 runtime for live/cache modes. Evaluation is CPU-safe. This notebook runs repository files; it does not embed the research implementation. GitHub and Drive authentication are independent. Set paths for the Google account mounted below. Do not save tokens or private outputs in this notebook.

In [31]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Clone or update GitHub
For a private repository, use a fine-grained GitHub token with read-only Contents access to this repository. The GitHub user must have repository access. Enter it only at the hidden prompt. It is passed to Git through a temporary askpass helper, never stored in the URL, notebook, or Git config. Authenticate separately on each new session/account.

In [32]:
from pathlib import Path
import os
import subprocess
import tempfile
import getpass

REPO_URL = "https://github.com/AhmedCode110/AC-MOT.git"
REPO = Path("/content/AC-MOT")

with tempfile.TemporaryDirectory() as tmp:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"

    token = getpass.getpass("GitHub token (hidden, session only): ")
    env["ACMOT_GH_TOKEN"] = token

    helper = Path(tmp) / "askpass"
    helper.write_text(
        '#!/usr/bin/env python3\n'
        'import os, sys\n'
        'print("x-access-token" if "Username" in sys.argv[1] else os.environ["ACMOT_GH_TOKEN"])\n'
    )
    helper.chmod(0o700)

    env["GIT_ASKPASS"] = str(helper)

    command = ["git", "-c", "credential.helper="]

    if (REPO / ".git").is_dir():
        result = subprocess.run(
            command + ["-C", str(REPO), "pull", "--ff-only", "origin", "main"],
            env=env,
            text=True,
            capture_output=True
        )
    else:
        result = subprocess.run(
            command + ["clone", "--branch", "main", REPO_URL, str(REPO)],
            env=env,
            text=True,
            capture_output=True
        )

    print("RETURN CODE:", result.returncode)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

    result.check_returncode()

print("Repo ready at:", REPO)

GitHub token (hidden, session only): ··········
RETURN CODE: 0
STDOUT:
Updating 98f9ec6..a8f60f1
Fast-forward
 scripts/speedtest_top3.py | 5 +++++
 1 file changed, 5 insertions(+)

STDERR:
From https://github.com/AhmedCode110/AC-MOT
 * branch            main       -> FETCH_HEAD
   98f9ec6..a8f60f1  main       -> origin/main

Repo ready at: /content/AC-MOT


In [33]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(REPO/'scripts/setup_trackeval.py'),'/content/TrackEval'],check=True)


CompletedProcess(args=['/usr/bin/python3', '/content/AC-MOT/scripts/setup_trackeval.py', '/content/TrackEval'], returncode=0)

## Central configuration
Default is evaluation of existing recordings. Edit all placeholder paths before running. For live mode load `configs/live.json` and point `frozen` at your existing verified selection; no new selection is invented. For CPU replay use mode `replay` and set a compatible cache. Cache generation is opt-in mode `cache`, development split only. v12 deliberately uses FP32; FP16/TensorRT would change its protocol.

In [34]:
import json, uuid, shutil
CONFIG_NAME = (REPO / 'configs' / 'active_config.txt').read_text().strip()
print("Active config:", CONFIG_NAME)
CFG=json.loads((REPO/'configs'/CONFIG_NAME).read_text())
# EDIT paths here or use your own JSON stored on Drive.
COPY_DATASET_TO_LOCAL=False
# A shared Drive folder must be accessible to this account; add a shortcut
# to My Drive if needed and confirm its actual mounted path.
DATASET_ON_DRIVE=Path(CFG['dataset'])
assert (DATASET_ON_DRIVE/'annotations').is_dir(), f'Check Drive access/path: {DATASET_ON_DRIVE}'
if COPY_DATASET_TO_LOCAL:
    local=Path('/content')/('acmot_dataset_'+uuid.uuid4().hex[:8])/DATASET_ON_DRIVE.name
    shutil.copytree(DATASET_ON_DRIVE,local) # unique destination; never overwrites
    CFG['dataset']=str(local)
RESULTS=Path(CFG['output_root'])
RESULTS.mkdir(parents=True,exist_ok=True)
probe=RESULTS/('.write_probe_'+uuid.uuid4().hex)
probe.write_text('probe');probe.unlink() # only this newly created probe
CONFIG_PATH=Path('/content')/('acmot_config_'+uuid.uuid4().hex+'.json')
CONFIG_PATH.write_text(json.dumps(CFG,indent=2))


Active config: speedtest_top3.json


1586

In [35]:
# Print Python/PyTorch/CUDA/GPU/Ultralytics and fail before GPU work if unsuitable.
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH),'--check'],check=True)


CompletedProcess(args=['/usr/bin/python3', '/content/AC-MOT/scripts/run.py', '--config', '/content/acmot_config_d66d2b6f3b7d48289569786f5e20e133.json', '--check'], returncode=0)

In [ ]:
# Explicit execution: outputs go to a unique folder under output_root on Drive.
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH)],check=True)
print('Results root on Drive:', CFG['output_root'])
# The runner prints the exact OUTPUT FOLDER, including when a run fails.


## Multiple Google accounts
Share the dataset folder with each Google account, and provide a writable results folder in that account's Drive (or a shared folder with write access). Verify actual paths after mounting; shared folder ownership and shortcuts do not guarantee a particular path. Obtain GitHub access independently. Opening this notebook does not grant either permission. Keep large data, caches, weights and results on Drive; Colab `/content` is temporary. Restart the runtime after dependency installation if previously imported packages conflict.